In [2]:
import pandas as pd

# 1. Load the Excel file
df = pd.read_excel("/content/[5442] EcoMoveMobility.xlsx")

# 2. Convert Date from Excel number to a real date
df["Date"] = pd.to_datetime(df["Date"], unit="D", origin="1899-12-30")

# 3. Add helpful columns
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.to_period("M").dt.to_timestamp()
df["Profit_EUR"] = df["Ticket_Revenue_EUR"] - df["Operating_Cost_EUR"]

# 4. Quick checks
print("Rows and columns:", df.shape)
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print(df.head())

Rows and columns: (5000, 21)
Date range: 2024-01-01 to 2025-12-31
Missing values: 0
Duplicate rows: 0
  Record_ID       Date       City      Country Transport_Mode  \
0  EM-00001 2025-03-31      Paris       France           Tram   
1  EM-00002 2025-03-01      Paris       France           Tram   
2  EM-00003 2025-08-13  Rotterdam  Netherlands           Tram   
3  EM-00004 2024-07-31       Lyon       France           Tram   
4  EM-00005 2024-10-13  Amsterdam  Netherlands          Metro   

          Route_Type Weather Event_Day  Passengers  Trips  ...  \
0       Tourist Zone  Cloudy       Yes       22946    101  ...   
1    University Area  Cloudy        No        9417     59  ...   
2  Airport Connector   Clear        No        6467     38  ...   
3    University Area   Clear       Yes        5218     33  ...   
4       Tourist Zone  Cloudy        No       10258     19  ...   

   On_Time_Percentage  Ticket_Revenue_EUR  Operating_Cost_EUR  CO2_Saved_KG  \
0                85.1          

In [4]:
import pandas as pd
import plotly.express as px

# Load and prepare (same as Step 1)
df = pd.read_excel("/content/[5442] EcoMoveMobility.xlsx")
df["Date"] = pd.to_datetime(df["Date"], unit="D", origin="1899-12-30")
df["Month"] = df["Date"].dt.to_period("M").dt.to_timestamp()

# Total passengers per month
monthly = df.groupby("Month")["Passengers"].sum().reset_index()

# Line chart
fig = px.line(monthly, x="Month", y="Passengers", markers=True,
              title="Monthly Passengers (2024-2025)")
fig.update_layout(xaxis_title="Month", yaxis_title="Total Passengers")
fig.show()

In [5]:
import pandas as pd
import plotly.express as px

# Load data
df = pd.read_excel("/content/[5442] EcoMoveMobility.xlsx")

# Total passengers per city, sorted smallest to largest
city = df.groupby("City")["Passengers"].sum().sort_values().reset_index()

# Horizontal bar chart
fig = px.bar(city, x="Passengers", y="City", orientation="h",
             title="Total Passengers by City", text="Passengers")
fig.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig.update_layout(xaxis_title="Total Passengers", yaxis_title="City")
fig.show()

In [7]:
import pandas as pd
import plotly.express as px

# Load data
df = pd.read_excel("/content/[5442] EcoMoveMobility.xlsx")

# Pick the numeric columns to compare
cols = ["Passengers", "Trips", "Average_Delay_Minutes", "On_Time_Percentage",
        "Ticket_Revenue_EUR", "Operating_Cost_EUR", "CO2_Saved_KG",
        "Energy_Consumption_KWh", "Customer_Rating", "Accessibility_Complaints"]

# Calculate correlations between every pair
corr = df[cols].corr().round(2)

# Heatmap
fig = px.imshow(corr, text_auto=True, color_continuous_scale="RdBu_r",
                zmin=-1, zmax=1, title="Correlation Heatmap of Key Measures")
fig.update_layout(width=800, height=700)
fig.show()

In [8]:
import pandas as pd
import plotly.express as px

# Load data
df = pd.read_excel("/content/[5442] EcoMoveMobility.xlsx")

# Box plot
fig = px.box(df, x="Transport_Mode", y="Average_Delay_Minutes",
             color="Transport_Mode", points="outliers",
             title="Delay Distribution by Transport Mode")
fig.update_layout(xaxis_title="Transport Mode",
                  yaxis_title="Average Delay (minutes)", showlegend=False)
fig.show()

# Summary numbers to support the chart
print(df.groupby("Transport_Mode")["Average_Delay_Minutes"]
        .describe().round(2).sort_values("50%", ascending=False).to_string())

                 count  mean   std  min  25%  50%  75%   max
Transport_Mode                                              
Bus             1375.0  6.58  1.58  1.2  5.5  6.5  7.5  12.3
Park and Ride    585.0  5.17  1.64  0.6  4.0  5.0  6.0  11.5
Tram             761.0  4.79  1.68  0.3  3.6  4.7  5.7  12.5
Metro           1047.0  3.73  1.60  0.0  2.7  3.6  4.6  10.1
E-Scooter        473.0  1.94  1.52  0.0  0.7  1.7  2.9   6.9
Bike Share       759.0  1.62  1.33  0.0  0.5  1.5  2.4   7.3


In [9]:
import pandas as pd
import plotly.graph_objects as go

# Load data
df = pd.read_excel("/content/[5442] EcoMoveMobility.xlsx")

# Revenue, cost and margin by transport mode
mode = df.groupby("Transport_Mode")[["Ticket_Revenue_EUR", "Operating_Cost_EUR"]].sum()
mode["Profit_EUR"] = mode["Ticket_Revenue_EUR"] - mode["Operating_Cost_EUR"]
mode["Margin_%"] = (mode["Profit_EUR"] / mode["Ticket_Revenue_EUR"] * 100).round(1)
mode = mode.sort_values("Margin_%")

# Dumbbell chart: a line joins cost to revenue for each mode
fig = go.Figure()
for m, row in mode.iterrows():
    fig.add_trace(go.Scatter(x=[row["Operating_Cost_EUR"], row["Ticket_Revenue_EUR"]],
                             y=[m, m], mode="lines",
                             line=dict(color="grey", width=3), showlegend=False))
fig.add_trace(go.Scatter(x=mode["Operating_Cost_EUR"], y=mode.index,
                         mode="markers", name="Operating Cost",
                         marker=dict(color="red", size=14)))
fig.add_trace(go.Scatter(x=mode["Ticket_Revenue_EUR"], y=mode.index,
                         mode="markers", name="Ticket Revenue",
                         marker=dict(color="green", size=14)))
fig.update_layout(title="Revenue vs Operating Cost by Transport Mode (EUR, log scale)",
                  xaxis_title="EUR (log scale)", yaxis_title="Transport Mode",
                  xaxis_type="log")
fig.show()

print(mode.round(0).to_string())

                Ticket_Revenue_EUR  Operating_Cost_EUR  Profit_EUR  Margin_%
Transport_Mode                                                              
Bike Share                310355.0           5912287.0  -5601932.0   -1805.0
E-Scooter                 261278.0           3536960.0  -3275681.0   -1254.0
Bus                     12002119.0          11992457.0      9661.0       0.0
Metro                   73334171.0          69029227.0   4304944.0       6.0
Tram                    15666168.0          13355947.0   2310221.0      15.0
Park and Ride           10066524.0           8283138.0   1783386.0      18.0


In [12]:
pip install dash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 23.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [dash]


In [13]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

# ---------- 1. LOAD AND PREPARE DATA ----------
df = pd.read_excel("/content/[5442] EcoMoveMobility.xlsx")
df["Date"] = pd.to_datetime(df["Date"], unit="D", origin="1899-12-30")
df["Month"] = df["Date"].dt.to_period("M").dt.to_timestamp()
df["Profit_EUR"] = df["Ticket_Revenue_EUR"] - df["Operating_Cost_EUR"]

# ---------- 2. LAYOUT ----------
app = Dash(__name__)

def dropdown(label, column):
    """Make one multi-select filter. Empty = show everything."""
    return html.Div([
        html.Label(label),
        dcc.Dropdown(id=column, options=sorted(df[column].unique()),
                     multi=True, placeholder="All")
    ], style={"width": "19%", "display": "inline-block", "marginRight": "1%"})

app.layout = html.Div([
    html.H2("EcoMove Sustainable Mobility Dashboard"),
    html.P("Use the filters to explore performance. Leave a filter empty to include everything."),

    # Filters
    html.Div([dropdown("City", "City"),
              dropdown("Transport Mode", "Transport_Mode"),
              dropdown("Route Type", "Route_Type"),
              dropdown("Weather", "Weather"),
              dropdown("Event Day", "Event_Day")]),

    # KPI cards
    html.Div(id="kpis", style={"display": "flex", "gap": "10px", "margin": "15px 0"}),

    # Charts: 2 per row
    html.Div([dcc.Graph(id="trend", style={"width": "50%"}),
              dcc.Graph(id="city_bar", style={"width": "50%"})], style={"display": "flex"}),
    html.Div([dcc.Graph(id="delay_box", style={"width": "50%"}),
              dcc.Graph(id="profit_bar", style={"width": "50%"})], style={"display": "flex"}),
    html.Div([dcc.Graph(id="access_heat", style={"width": "50%"}),
              dcc.Graph(id="co2_scatter", style={"width": "50%"})], style={"display": "flex"}),
], style={"fontFamily": "Arial", "padding": "20px"})

# ---------- 3. KPI CARD HELPER ----------
def card(title, value):
    return html.Div([html.Div(title, style={"fontSize": "13px", "color": "grey"}),
                     html.Div(value, style={"fontSize": "22px", "fontWeight": "bold"})],
                    style={"flex": "1", "padding": "10px", "border": "1px solid #ddd",
                           "borderRadius": "6px", "textAlign": "center"})

# ---------- 4. CALLBACK: filters -> all charts ----------
@app.callback(
    [Output("kpis", "children"), Output("trend", "figure"), Output("city_bar", "figure"),
     Output("delay_box", "figure"), Output("profit_bar", "figure"),
     Output("access_heat", "figure"), Output("co2_scatter", "figure")],
    [Input(c, "value") for c in ["City", "Transport_Mode", "Route_Type", "Weather", "Event_Day"]])
def update(city, mode, route, weather, event):
    # Filter the data (skip any filter left empty)
    d = df.copy()
    for col, chosen in [("City", city), ("Transport_Mode", mode), ("Route_Type", route),
                        ("Weather", weather), ("Event_Day", event)]:
        if chosen:
            d = d[d[col].isin(chosen)]

    if d.empty:
        empty = px.scatter(title="No data for this filter combination")
        return [card("No data", "-")], empty, empty, empty, empty, empty, empty

    # KPI cards
    pax = d["Passengers"].sum()
    kpis = [card("Passengers", f"{pax/1e6:.2f}M"),
            card("Profit", f"€{d['Profit_EUR'].sum()/1e6:.2f}M"),
            card("Avg Delay", f"{d['Average_Delay_Minutes'].mean():.1f} min"),
            card("On-Time", f"{d['On_Time_Percentage'].mean():.1f}%"),
            card("Rating", f"{d['Customer_Rating'].mean():.2f} / 5"),
            card("CO2 Saved", f"{d['CO2_Saved_KG'].sum()/1e6:.2f} kt"),
            card("Complaints / 1k pax", f"{d['Accessibility_Complaints'].sum()/pax*1000:.2f}")]

    # Chart 1: demand trend (line)
    m = d.groupby("Month")["Passengers"].sum().reset_index()
    trend = px.line(m, x="Month", y="Passengers", markers=True, title="Demand Trend")

    # Chart 2: passengers by city (bar)
    c = d.groupby("City")["Passengers"].sum().sort_values().reset_index()
    city_bar = px.bar(c, x="Passengers", y="City", orientation="h", title="Passengers by City")

    # Chart 3: delay by mode (box)
    delay_box = px.box(d, x="Transport_Mode", y="Average_Delay_Minutes",
                       color="Transport_Mode", title="Delay Distribution by Mode")
    delay_box.update_layout(showlegend=False)

    # Chart 4: profit by mode (bar, green = profit, red = loss)
    p = d.groupby("Transport_Mode")["Profit_EUR"].sum().reset_index()
    p["Result"] = p["Profit_EUR"].apply(lambda x: "Profit" if x >= 0 else "Loss")
    profit_bar = px.bar(p, x="Transport_Mode", y="Profit_EUR", color="Result",
                        color_discrete_map={"Profit": "green", "Loss": "red"},
                        title="Profit by Transport Mode (EUR)")

    # Chart 5: accessibility complaints per 1,000 passengers (heatmap)
    a = d.groupby(["City", "Route_Type"])[["Accessibility_Complaints", "Passengers"]].sum()
    a["Rate"] = a["Accessibility_Complaints"] / a["Passengers"] * 1000
    a = a["Rate"].unstack().round(2)
    access_heat = px.imshow(a, text_auto=True, color_continuous_scale="Reds", aspect="auto",
                            title="Accessibility Complaints per 1,000 Passengers")

    # Chart 6: CO2 saved vs energy used (scatter)
    co2_scatter = px.scatter(d, x="Energy_Consumption_KWh", y="CO2_Saved_KG",
                             color="Transport_Mode", opacity=0.6,
                             title="CO2 Saved vs Energy Consumption")

    return kpis, trend, city_bar, delay_box, profit_bar, access_heat, co2_scatter

# ---------- 5. RUN ----------
if __name__ == "__main__":
    app.run(debug=False)

<IPython.core.display.Javascript object>